# QDMpy Fitting Tutorial

In this tutorial, we'll explore the fitting capabilities of QDMpy, with a focus on:

1. Understanding ODMR spectral models
2. Working with the FitManager class
3. Using and configuring the ConstraintManager
4. Advanced fitting techniques and customization

This tutorial demonstrates how to use QDMpy's built-in fitting functionality to analyze ODMR spectra from NV centers in diamond. We'll focus on the recently implemented ConstraintManager, which provides a robust way to set and manage constraints on fit parameters.

## 1. Setup and Imports

First, let's import the necessary modules:

In [1]:
import sys
sys.path.append("/home/mike/git/QDMpy/src")

In [2]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import QDMpy
from QDMpy.fit import FitManager, ConstraintManager, CONSTRAINT_TYPES
from QDMpy.models import ModelRegistry

# Set up logging
import logging
logging.basicConfig(level=logging.INFO)

# Print QDMpy version
print(f"QDMpy version: {QDMpy.__version__}")

14:56:45.769     INFO QDMpy.<module> >> WELCOME TO QDMpy
14:56:45.769    DEBUG QDMpy.<module> >> QDMpy version 0.1.0a installed at /home/mike/git/QDMpy/src/QDMpy
14:56:45.770    DEBUG QDMpy.<module> >> QDMpy config file /home/mike/.config/QDMpy/config.ini
14:56:45.770     INFO QDMpy.load_config >> Loading config file: /home/mike/.config/QDMpy/config.ini
0 <class 'ctypes.c_ulong'>
1 <class 'ctypes.c_ulong'>
2 <class 'pygpufit.gpufit.LP_c_float'>
3 <class 'pygpufit.gpufit.LP_c_float'>
4 <class 'ctypes.c_int'>
5 <class 'pygpufit.gpufit.LP_c_float'>
6 <class 'pygpufit.gpufit.LP_c_float'>
7 <class 'pygpufit.gpufit.LP_c_int'>
8 <class 'ctypes.c_float'>
9 <class 'ctypes.c_int'>
10 <class 'pygpufit.gpufit.LP_c_int'>
11 <class 'ctypes.c_int'>
12 <class 'ctypes.c_ulong'>
13 <class 'ctypes.LP_c_char'>
14 <class 'pygpufit.gpufit.LP_c_float'>
15 <class 'pygpufit.gpufit.LP_c_int'>
16 <class 'pygpufit.gpufit.LP_c_float'>
17 <class 'pygpufit.gpufit.LP_c_int'>
14:56:45.772     INFO QDMpy.<module> >> CU

## 2. Understanding ODMR Spectral Models

QDMpy includes several spectral models for fitting ODMR data from NV centers in diamond. Let's examine the available models:

In [3]:
# List all available models
available_models = ModelRegistry.all()
print(f"Available models: {list(available_models.keys())}")

# Look at details of a specific model
esrsingle = ModelRegistry.get("ESRSINGLE")
print(f"\nESRSINGLE model details:")
print(f"  Parameters: {esrsingle.parameter}")
print(f"  Unique parameters: {esrsingle.parameters_unique}")
print(f"  Number of parameters: {esrsingle.n_parameters}")

# Check another model with more resonances
esr14n = ModelRegistry.get("ESR14N")
print(f"\nESR14N model details:")
print(f"  Parameters: {esr14n.parameter}")
print(f"  Unique parameters: {esr14n.parameters_unique}")
print(f"  Number of parameters: {esr14n.n_parameters}")

Available models: ['ESR14N', 'ESR15N', 'ESRSINGLE']

ESRSINGLE model details:
  Parameters: ['contrast', 'center', 'width', 'offset']
  Unique parameters: ['contrast', 'center', 'width_0', 'offset']
  Number of parameters: 4

ESR14N model details:
  Parameters: ['contrast', 'center', 'width', 'width', 'width', 'offset']
  Unique parameters: ['contrast', 'center', 'width_0', 'width_1', 'width_2', 'offset']
  Number of parameters: 6


### Model Types Explained

QDMpy supports three main model types:

- **ESRSINGLE**: A single Lorentzian dip for simple ODMR spectra
- **ESR14N**: Three Lorentzian dips for nitrogen-14 NV centers (hyperfine splitting)
- **ESR15N**: Two Lorentzian dips for nitrogen-15 NV centers (hyperfine splitting)

Each model has a set of parameters:
- **center**: The central frequency of the resonance (in GHz)
- **width**: The width of the resonance (in GHz)
- **contrast**: The depth of the resonance dip (unitless)
- **offset**: The baseline value of the spectrum (unitless)

For multiple resonance models, the parameters are indexed (e.g., center_0, center_1, etc.).

## 3. Creating Synthetic ODMR Data

Let's generate some synthetic ODMR data to demonstrate the fitting process:

In [ ]:
# Create frequency axis
frequencies = np.linspace(2.87e9, 2.88e9, 100)  # 100 points from 2.87 to 2.88 GHz

# Use QDMpy's built-in models to generate synthetic data
# First, get the ESRSINGLE model
esrsingle_model = ModelRegistry.get('ESRSINGLE')

# Define base parameters
center = 2.875e9  # Center frequency in Hz
width = 5e6       # Width in Hz
contrast = 0.1    # Contrast depth (0-1)
offset = 1.0      # Baseline offset
noise_level = 0.005  # Noise level

# Generate data for 4 different pixels with slight variations
n_pixels = 4
data = np.ones((2, 1, 100, n_pixels))  # 2 polarities, 1 frequency range, 100 frequency points, 4 pixels

for pol in range(2):  # For each polarity
    for pixel in range(n_pixels):
        # Vary parameters slightly for each pixel
        pixel_center = center + (pixel - 1.5) * 1e6  # Shift center frequency
        pixel_width = width * (0.8 + 0.1 * pixel)  # Vary width
        pixel_contrast = contrast * (0.8 + 0.1 * pixel)  # Vary contrast
        
        # Create parameter array for this pixel
        # Parameters must be in the order expected by the model (contrast, center, width, offset)
        params = np.array([pixel_contrast, pixel_center, pixel_width, offset], dtype=np.float32)
        
        # Generate clean spectrum using the model's equation
        # For ESRSINGLE: offset - contrast * (width^2 / ((f - center)^2 + width^2))
        x_vals = frequencies
        clean_spectrum = offset - pixel_contrast * (pixel_width**2 / ((x_vals - pixel_center)**2 + pixel_width**2))
        
        # Add noise
        noise = np.random.normal(0, noise_level, len(frequencies))
        data[pol, 0, :, pixel] = clean_spectrum + noise

# Plot the synthetic data
plt.figure(figsize=(10, 6))
for pixel in range(n_pixels):
    plt.plot(frequencies/1e9, data[0, 0, :, pixel], label=f'Pixel {pixel}')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Signal (a.u.)')
plt.title('Synthetic ODMR Spectra')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Introduction to the FitManager

The `FitManager` is the main class for fitting ODMR spectra in QDMpy. It handles:

- Model selection and management
- Parameter estimation (guessing initial values)
- Constraint management
- Fitting execution
- Result organization and access

Let's create a FitManager instance with our synthetic data:

In [ ]:
# Create FitManager with our synthetic data
fit_manager = FitManager(data, frequencies)

# Display information about the fit manager
print(fit_manager)
print(f"Model name: {fit_manager.model_name}")
print(f"Model parameters: {fit_manager.model_params}")
print(f"Unique parameters: {fit_manager.model_params_unique}")

### Examining the Initial Parameter Guesses

QDMpy automatically estimates initial parameter values based on the data:

In [ ]:
# Get initial parameter guesses
initial_params = fit_manager.get_initial_parameter()
print(f"Initial parameter shape: {initial_params.shape}")

# Print initial parameter values for the first pixel
print("\nInitial parameters for pixel 0:")
for i, param in enumerate(fit_manager.model_params_unique):
    value = initial_params[0, 0, 0, i]
    print(f"  {param}: {value}")

# Compare with actual values
print("\nActual values for pixel 0:")
print(f"  center: {center - 1.5e6}")
print(f"  width: {width * 0.8}")
print(f"  contrast: {contrast * 0.8}")
print(f"  offset: {offset}")

## 5. Understanding the ConstraintManager

The `ConstraintManager` is responsible for managing the constraints on fit parameters. Constraints help stabilize the fitting process and ensure physically meaningful results.

Let's examine the default constraints:

In [ ]:
# View current constraints
constraints = fit_manager.constraints
print("Default constraints:")
for param, constraint in constraints.items():
    print(f"  {param}:")
    print(f"    Min: {constraint[0]}")
    print(f"    Max: {constraint[1]}")
    print(f"    Type: {constraint[2]}")
    print(f"    Unit: {constraint[3]}")

### Constraint Types

QDMpy supports four constraint types:

1. **FREE**: No constraints (parameters can take any value)
2. **LOWER**: Only a lower bound is applied
3. **UPPER**: Only an upper bound is applied
4. **LOWER_UPPER**: Both lower and upper bounds are applied

Let's modify some constraints to demonstrate how this works:

In [ ]:
# Set more restrictive constraints on center frequency
fit_manager.set_constraints(
    'center', 
    vmin=2.874e9,     # Lower bound (in Hz)
    vmax=2.876e9,     # Upper bound (in Hz)
    constraint_type='LOWER_UPPER'
)

# Set a minimum width to avoid unrealistically narrow features
fit_manager.set_constraints(
    'width_0', 
    vmin=2e6,         # Lower bound (in Hz)
    constraint_type='LOWER'
)

# Set a maximum contrast to avoid over-fitting
fit_manager.set_constraints(
    'contrast', 
    vmax=0.2,         # Upper bound
    constraint_type='UPPER'
)

# Print updated constraints
print("Updated constraints:")
for param, constraint in fit_manager.constraints.items():
    print(f"  {param}:")
    print(f"    Min: {constraint[0]}")
    print(f"    Max: {constraint[1]}")
    print(f"    Type: {constraint[2]}")

### Internal Representation for pyGpufit

Internally, the ConstraintManager converts these constraints into formats suitable for the GPU fitting library:

In [ ]:
# Get constraints in array format
constraints_array = fit_manager.get_constraints_array(2)  # For 2 pixels
print(f"Constraints array shape: {constraints_array.shape}")
print(f"Constraints array first row:\n{constraints_array[0]}")

# Get constraint types as indices
constraint_types = fit_manager.get_constraint_types()
print(f"\nConstraint types: {constraint_types}")
print(f"Type names: {[CONSTRAINT_TYPES[i] for i in constraint_types]}")

## The Fitting Process

Now, let's perform fitting on our synthetic data. Note that this requires pyGpufit to be installed.

In [ ]:
try:
    # Attempt to fit the data
    fit_manager.fit_odmr()
    fitting_successful = True
    print("Fitting completed successfully!")
except ImportError:
    fitting_successful = False
    print("Warning: pyGpufit is required for fitting but not installed.")
    print("This part of the tutorial will be simulated.")

### Accessing and Visualizing Fit Results

If fitting is successful, we can examine the results:

In [ ]:
if fitting_successful:
    # Get fitted parameters
    fit_params = fit_manager.parameter
    
    # Print fitted parameters for the first pixel
    print("Fitted parameters for pixel 0:")
    for i, param in enumerate(fit_manager.model_params_unique):
        value = fit_params[0, 0, 0, i]
        print(f"  {param}: {value}")
    
    # Get specific parameters
    centers = fit_manager.get_param('center')
    widths = fit_manager.get_param('width_0')
    contrasts = fit_manager.get_param('contrast')
    chi_squares = fit_manager.get_param('chi2')
    
    print("\nFit quality (chi-square values):")
    print(chi_squares)
else:
    # Simulate results for demonstration
    print("Simulated fit results for demonstration:")
    for i, param in enumerate(fit_manager.model_params_unique):
        # Get true value with small perturbation
        if param == 'center':
            value = center - 1.5e6 + np.random.normal(0, 1e5)
        elif param == 'width_0':
            value = width * 0.8 + np.random.normal(0, 1e5)
        elif param == 'contrast':
            value = contrast * 0.8 + np.random.normal(0, 0.01)
        else:  # offset
            value = offset + np.random.normal(0, 0.01)
        print(f"  {param}: {value}")

In [ ]:
# Plot fitted spectra (if fitting was successful) or simulated fits
plt.figure(figsize=(12, 8))

# Plot data for first pixel and polarity
pixel_idx = 0
plt.plot(frequencies/1e9, data[0, 0, :, pixel_idx], 'o', markersize=3, alpha=0.7, label='Observed data')

# Generate fitted curve
if fitting_successful:
    # Use actual fit parameters
    fit_params = fit_manager.parameter[0, 0, pixel_idx]
    center_val = fit_params[fit_manager.model_params_unique.index('center')]
    width_val = fit_params[fit_manager.model_params_unique.index('width_0')]
    contrast_val = fit_params[fit_manager.model_params_unique.index('contrast')]
    offset_val = fit_params[fit_manager.model_params_unique.index('offset')]
else:
    # Use simulated parameters
    center_val = center - 1.5e6 + np.random.normal(0, 1e5)
    width_val = width * 0.8 + np.random.normal(0, 1e5)
    contrast_val = contrast * 0.8 + np.random.normal(0, 0.01)
    offset_val = offset + np.random.normal(0, 0.01)

# Generate smooth curve using the model
f_fine = np.linspace(frequencies.min(), frequencies.max(), 500)
params = np.array([contrast_val, center_val, width_val, offset_val], dtype=np.float32)

# Use the model equation directly for visualization
fit_curve = offset_val - contrast_val * (width_val**2 / ((f_fine - center_val)**2 + width_val**2))

plt.plot(f_fine/1e9, fit_curve, '-', linewidth=2, label='Fitted curve')

# Add annotations with fit parameters
plt.text(0.02, 0.15, 
         f"Center: {center_val/1e9:.6f} GHz\n"
         f"Width: {width_val/1e6:.3f} MHz\n"
         f"Contrast: {contrast_val:.4f}\n"
         f"Offset: {offset_val:.4f}", 
         transform=plt.gca().transAxes, 
         bbox=dict(facecolor='white', alpha=0.8))

plt.xlabel('Frequency (GHz)')
plt.ylabel('Signal (a.u.)')
plt.title('ODMR Data with Fit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Advanced Constraint Management

For more complex models like ESR14N or ESR15N, constraint management becomes more important. Let's explore some advanced techniques:

In [ ]:
# Create a new FitManager with a more complex model
complex_fit = FitManager(data, frequencies, model_name='ESR14N')
print(f"Model parameters: {complex_fit.model_params}")
print(f"Unique parameters: {complex_fit.model_params_unique}")

# Setup for the complex model
# We can generate a 14N spectrum example to see what it looks like
esr14n_model = ModelRegistry.get('ESR14N')
center_base = 2.875e9
hyperfine = 2.2e6  # 2.2 MHz hyperfine splitting for 14N
contrast_val = 0.05
width_val = 5e6
offset_val = 1.0

# Create parameter array for ESR14N model
# Note: ESR14N model parameters are [contrast, center, width, width, width, offset]
# or mapped to unique parameters: [contrast, center, width_0, width_1, width_2, offset]
params = np.array([contrast_val, center_base, width_val, width_val, width_val, offset_val], dtype=np.float32)

# Generate ESR14N spectrum
f_fine = np.linspace(frequencies.min(), frequencies.max(), 500)
esr14n_spectrum = offset_val

# Manually calculate the three peaks with hyperfine splitting
for i, shift in enumerate([-hyperfine, 0, hyperfine]):
    center_shifted = center_base + shift
    esr14n_spectrum -= contrast_val * (width_val**2 / ((f_fine - center_shifted)**2 + width_val**2))

# Plot 14N spectrum
plt.figure(figsize=(10, 6))
plt.plot(f_fine/1e9, esr14n_spectrum, linewidth=2)
plt.xlabel('Frequency (GHz)')
plt.ylabel('Signal (a.u.)')
plt.title('ESR14N Model Spectrum (Three Resonances)')
plt.grid(True, alpha=0.3)
plt.show()

# Now set constraints for the ESR14N model
# Set tight constraints on the three resonance centers based on hyperfine splitting
complex_fit.set_constraints('center', vmin=center_base-1e6, vmax=center_base+1e6, constraint_type='LOWER_UPPER')

# Ensure all resonances have similar widths
for i in range(3):
    complex_fit.set_constraints(f'width_{i}', vmin=3e6, vmax=7e6, constraint_type='LOWER_UPPER')

# View updated constraints
print("\nConstraints for ESR14N model:")
for param in ['center', 'width_0', 'width_1', 'width_2']:
    constraint = complex_fit.constraints[param]
    print(f"  {param}: [{constraint[0]/1e6:.2f} - {constraint[1]/1e6:.2f}] MHz, Type: {constraint[2]}")

### Removing All Constraints

Sometimes, you might want to allow completely free fitting without any constraints:

In [ ]:
# Set all parameters to unconstrained fitting
complex_fit.set_free_constraints()

# Verify constraints are set to FREE
print("Constraints after setting to FREE:")
for param in complex_fit.model_params_unique:
    assert complex_fit.constraints[param][2] == 'FREE'
    print(f"  {param}: {complex_fit.constraints[param][2]}")

## Summary

In this tutorial, we've explored:

1. **ODMR Spectral Models**: Understanding the different models available for fitting NV center ODMR spectra
2. **FitManager**: Working with the main fitting class to handle model selection, parameter estimation, and fitting
3. **ConstraintManager**: Using and configuring constraints to ensure stable and physically meaningful fits
4. **Advanced Techniques**: Working with complex models and specialized constraints

The constraint management system in QDMpy provides powerful control over the fitting process, allowing you to incorporate domain knowledge and physical constraints into your analysis pipeline.